# AI Ops Automation Notebook

End-to-end AI Ops pipeline: Jira ingestion → classification → log relevance detection → semantic retrieval → RCA generation → notifications.

## Model Registry

| Model | Task |
|---|---|
| DeBERTa-v3-base | Ticket Classification |
| TF-IDF + Logistic Regression | Classification Fallback |
| LightGBM (Multiclass) | Priority Prediction (P1–P5) |
| DistilRoBERTa-base | Emotion Detection |
| MiniLM (all-MiniLM-L6-v2) | Log Relevance Detection |
| BGE-base-en-v1.5 + FAISS | Semantic Retrieval (RAG-style) |
| LightGBM (Binary) | Incident Risk Prediction |
| Orchestration Layer | Model Management |
| Rule-Based RCA Generator | Explainable RCA |

## Notebook Structure
1. Install Dependencies
2. Configuration
3. Model Loader (Orchestration Layer)
4. Ticket Classification — DeBERTa-v3-base + TF-IDF/LR Fallback
5. Emotion Detection — DistilRoBERTa-base
6. Priority Prediction — LightGBM Multiclass
7. Incident Risk Prediction — LightGBM Binary
8. Log Relevance Detection — MiniLM (all-MiniLM-L6-v2)
9. Semantic Retrieval — BGE-base-en-v1.5 + FAISS
10. Rule-Based RCA Generator
11. Notification
12. Confluence KB
13. FastAPI Application
14. Manual Pipeline Trigger

## 1. Install Dependencies

In [ ]:
# Uncomment and run to install all required packages
# !pip install \
#     transformers torch \
#     sentence-transformers faiss-cpu \
#     lightgbm scikit-learn \
#     fastapi uvicorn python-dotenv \
#     requests azure-storage-file-share \
#     joblib numpy

## 2. Configuration

Central configuration loaded from environment variables.

In [ ]:
"""Central configuration loaded from environment variables."""

from __future__ import annotations

import logging
import os

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("aiops")

# ---------------------------------------------------------------------------
# Jira
# ---------------------------------------------------------------------------
JIRA_BASE_URL: str = os.getenv("JIRA_BASE_URL", "")          # e.g. https://myorg.atlassian.net
JIRA_USER: str = os.getenv("JIRA_USER", "")                   # service-account email
JIRA_API_TOKEN: str = os.getenv("JIRA_API_TOKEN", "")
JIRA_PROJECT_KEY: str = os.getenv("JIRA_PROJECT_KEY", "OPS")

# ---------------------------------------------------------------------------
# Confluence
# ---------------------------------------------------------------------------
CONFLUENCE_BASE_URL: str = os.getenv("CONFLUENCE_BASE_URL", JIRA_BASE_URL)
CONFLUENCE_SPACE_KEY: str = os.getenv("CONFLUENCE_SPACE_KEY", "KB")
CONFLUENCE_PARENT_PAGE_ID: str = os.getenv("CONFLUENCE_PARENT_PAGE_ID", "")

# ---------------------------------------------------------------------------
# Azure Storage (mobile logs)
# ---------------------------------------------------------------------------
AZURE_STORAGE_CONNECTION_STRING: str = os.getenv("AZURE_STORAGE_CONNECTION_STRING", "")
AZURE_FILE_SHARE_NAME: str = os.getenv("AZURE_FILE_SHARE_NAME", "mobile-logs")
AZURE_LOG_DIRECTORY: str = os.getenv("AZURE_LOG_DIRECTORY", "")

# ---------------------------------------------------------------------------
# Datadog (server logs + monitoring)
# ---------------------------------------------------------------------------
DATADOG_API_KEY: str = os.getenv("DATADOG_API_KEY", "")
DATADOG_APP_KEY: str = os.getenv("DATADOG_APP_KEY", "")
DATADOG_SITE: str = os.getenv("DATADOG_SITE", "datadoghq.com")
DATADOG_LOG_LOOKBACK_HOURS: int = int(os.getenv("DATADOG_LOG_LOOKBACK_HOURS", "6"))

# ---------------------------------------------------------------------------
# OpenAI / Azure OpenAI (kept for optional LLM enhancement)
# ---------------------------------------------------------------------------
OPENAI_API_KEY: str = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL: str = os.getenv("OPENAI_MODEL", "gpt-4o")
AZURE_OPENAI_ENDPOINT: str = os.getenv("AZURE_OPENAI_ENDPOINT", "")
AZURE_OPENAI_API_KEY: str = os.getenv("AZURE_OPENAI_API_KEY", "")
AZURE_OPENAI_DEPLOYMENT: str = os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4o")
AZURE_OPENAI_API_VERSION: str = os.getenv("AZURE_OPENAI_API_VERSION", "2024-02-01")

# ---------------------------------------------------------------------------
# Email / SMTP
# ---------------------------------------------------------------------------
SMTP_HOST: str = os.getenv("SMTP_HOST", "smtp.gmail.com")
SMTP_PORT: int = int(os.getenv("SMTP_PORT", "587"))
SMTP_USER: str = os.getenv("SMTP_USER", "")
SMTP_PASSWORD: str = os.getenv("SMTP_PASSWORD", "")
ALERT_EMAIL_RECIPIENTS: list[str] = [
    e.strip()
    for e in os.getenv("ALERT_EMAIL_RECIPIENTS", "").split(",")
    if e.strip()
]

# ---------------------------------------------------------------------------
# Celery / Redis
# ---------------------------------------------------------------------------
REDIS_URL: str = os.getenv("REDIS_URL", "redis://localhost:6379/0")

# ---------------------------------------------------------------------------
# Model artifacts directory
# ---------------------------------------------------------------------------
ARTIFACTS_DIR: str = os.getenv("ARTIFACTS_DIR", "artifacts")

# ---------------------------------------------------------------------------
# Log alert thresholds
# ---------------------------------------------------------------------------
ERROR_COUNT_THRESHOLD: int = int(os.getenv("ERROR_COUNT_THRESHOLD", "50"))
WARNING_COUNT_THRESHOLD: int = int(os.getenv("WARNING_COUNT_THRESHOLD", "200"))
FATAL_COUNT_THRESHOLD: int = int(os.getenv("FATAL_COUNT_THRESHOLD", "5"))

# ---------------------------------------------------------------------------
# Retrieval
# ---------------------------------------------------------------------------
TOP_K_SIMILAR_TICKETS: int = int(os.getenv("TOP_K_SIMILAR_TICKETS", "5"))
TOP_K_LOG_LINES: int = int(os.getenv("TOP_K_LOG_LINES", "20"))
TOP_K_CONFLUENCE: int = int(os.getenv("TOP_K_CONFLUENCE", "3"))

# FAISS index path
FAISS_INDEX_PATH: str = os.getenv("FAISS_INDEX_PATH", "/tmp/aiops_faiss")

# ---------------------------------------------------------------------------
# HuggingFace model names
# ---------------------------------------------------------------------------
DEBERTA_MODEL_NAME: str = os.getenv("DEBERTA_MODEL_NAME", "microsoft/deberta-v3-base")
EMOTION_MODEL_NAME: str = os.getenv("EMOTION_MODEL_NAME", "j-hartmann/emotion-english-distilroberta-base")
MINILM_MODEL_NAME: str = os.getenv("MINILM_MODEL_NAME", "sentence-transformers/all-MiniLM-L6-v2")
BGE_MODEL_NAME: str = os.getenv("BGE_MODEL_NAME", "BAAI/bge-base-en-v1.5")

print("Configuration loaded.")

## 3. Model Loader — Orchestration Layer

Loads all model artifacts once at startup and manages them through a central registry.

**Models loaded:**
- `DeBERTa-v3-base` — ticket classification
- `TF-IDF + Logistic Regression` — classification fallback
- `LightGBM (Multiclass)` — priority prediction
- `LightGBM (Binary)` — incident risk prediction
- `DistilRoBERTa-base` — emotion detection
- `all-MiniLM-L6-v2` — log relevance detection
- `BGE-base-en-v1.5 + FAISS` — semantic retrieval

In [ ]:
"""Orchestration Layer: load and manage all model artifacts."""

from pathlib import Path
from typing import Any
import joblib

_REGISTRY: dict[str, Any] = {}


def _art(name: str) -> Path:
    return Path(ARTIFACTS_DIR) / name


# ---------------------------------------------------------------------------
# DeBERTa-v3-base — Ticket Classification
# ---------------------------------------------------------------------------

def _load_deberta_classifier() -> None:
    """Load fine-tuned DeBERTa-v3-base for ticket category classification."""
    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification

        path = _art("classification_deberta")
        # Use local fine-tuned checkpoint if available, else load from HuggingFace Hub
        model_path = str(path) if path.exists() else DEBERTA_MODEL_NAME

        tok = AutoTokenizer.from_pretrained(model_path)
        model = AutoModelForSequenceClassification.from_pretrained(model_path)
        model.eval()

        _REGISTRY["deberta_tokenizer"] = tok
        _REGISTRY["deberta_model"] = model

        # Label encoder (required for fine-tuned local checkpoint)
        le_path = path / "label_encoder.joblib"
        if le_path.exists():
            _REGISTRY["cat_le"] = joblib.load(le_path)

        logger.info("DeBERTa-v3-base classifier loaded from %s", model_path)
    except Exception as exc:
        logger.warning("Could not load DeBERTa classifier: %s", exc)


# ---------------------------------------------------------------------------
# TF-IDF + Logistic Regression — Classification Fallback
# ---------------------------------------------------------------------------

def _load_tfidf_lr_fallback() -> None:
    """Load TF-IDF vectorizer + Logistic Regression as classification fallback."""
    try:
        path = _art("tfidf_lr_fallback")
        if not path.exists():
            logger.warning("TF-IDF/LR fallback not found at %s — skipping", path)
            return

        _REGISTRY["tfidf_vectorizer"] = joblib.load(path / "tfidf_vectorizer.joblib")
        _REGISTRY["lr_classifier"] = joblib.load(path / "lr_classifier.joblib")
        _REGISTRY["lr_label_encoder"] = joblib.load(path / "label_encoder.joblib")
        logger.info("TF-IDF + Logistic Regression fallback loaded from %s", path)
    except Exception as exc:
        logger.warning("Could not load TF-IDF/LR fallback: %s", exc)


# ---------------------------------------------------------------------------
# LightGBM Multiclass — Priority Prediction (P1–P5)
# ---------------------------------------------------------------------------

def _load_priority_lgbm() -> None:
    """Load LightGBM multiclass booster for priority prediction (P1–P5)."""
    try:
        import lightgbm as lgb

        path = _art("priority_lgbm")
        if not path.exists():
            logger.warning("Priority LightGBM model not found at %s — skipping", path)
            return

        _REGISTRY["prio_booster"] = lgb.Booster(model_file=str(path / "priority_model.txt"))
        _REGISTRY["prio_tfidf"] = joblib.load(path / "tfidf_vectorizer.joblib")
        _REGISTRY["prio_le"] = joblib.load(path / "priority_label_encoder.joblib")
        ohe_path = path / "ohe_encoder.joblib"
        _REGISTRY["prio_ohe"] = joblib.load(ohe_path) if ohe_path.exists() else None
        logger.info("LightGBM (Multiclass) priority model loaded from %s", path)
    except Exception as exc:
        logger.warning("Could not load priority LightGBM: %s", exc)


# ---------------------------------------------------------------------------
# LightGBM Binary — Incident Risk Prediction
# ---------------------------------------------------------------------------

def _load_incident_risk_lgbm() -> None:
    """Load LightGBM binary booster for incident risk prediction."""
    try:
        import lightgbm as lgb

        path = _art("incident_risk")
        if not path.exists():
            logger.warning("Incident risk model not found at %s — skipping", path)
            return

        _REGISTRY["incident_booster"] = lgb.Booster(
            model_file=str(path / "incident_risk_model.txt")
        )
        logger.info("LightGBM (Binary) incident risk model loaded from %s", path)
    except Exception as exc:
        logger.warning("Could not load incident risk LightGBM: %s", exc)


# ---------------------------------------------------------------------------
# DistilRoBERTa-base — Emotion Detection
# ---------------------------------------------------------------------------

def _load_emotion_detector() -> None:
    """Load DistilRoBERTa-base for emotion detection on ticket text."""
    try:
        from transformers import pipeline

        path = _art("emotion_distilroberta")
        model_path = str(path) if path.exists() else EMOTION_MODEL_NAME

        _REGISTRY["emotion_pipeline"] = pipeline(
            "text-classification",
            model=model_path,
            top_k=None,           # return all emotion scores
            truncation=True,
            max_length=512,
        )
        logger.info("DistilRoBERTa-base emotion detector loaded from %s", model_path)
    except Exception as exc:
        logger.warning("Could not load emotion detector: %s", exc)


# ---------------------------------------------------------------------------
# all-MiniLM-L6-v2 — Log Relevance Detection
# ---------------------------------------------------------------------------

def _load_minilm_log_ranker() -> None:
    """Load all-MiniLM-L6-v2 SentenceTransformer for log relevance detection."""
    try:
        from sentence_transformers import SentenceTransformer

        path = _art("minilm_log_ranker")
        model_path = str(path) if path.exists() else MINILM_MODEL_NAME

        _REGISTRY["minilm_model"] = SentenceTransformer(model_path)
        logger.info("all-MiniLM-L6-v2 log ranker loaded from %s", model_path)
    except Exception as exc:
        logger.warning("Could not load MiniLM log ranker: %s", exc)


# ---------------------------------------------------------------------------
# BGE-base-en-v1.5 + FAISS — Semantic Retrieval (RAG-style)
# ---------------------------------------------------------------------------

def _load_bge_faiss() -> None:
    """Load BGE-base-en-v1.5 embedder and FAISS index for semantic retrieval."""
    try:
        from sentence_transformers import SentenceTransformer
        import faiss
        import numpy as np

        path = _art("bge_faiss")
        model_path = str(path / "bge_model") if (path / "bge_model").exists() else BGE_MODEL_NAME

        _REGISTRY["bge_embedder"] = SentenceTransformer(model_path)

        index_file = path / "faiss_index.bin"
        meta_file = path / "faiss_metadata.joblib"
        if index_file.exists():
            _REGISTRY["faiss_index"] = faiss.read_index(str(index_file))
            _REGISTRY["faiss_metadata"] = joblib.load(meta_file) if meta_file.exists() else []
            logger.info("BGE-base-en-v1.5 + FAISS index loaded from %s", path)
        else:
            # Bootstrap empty FAISS index (dim=768 for BGE-base)
            _REGISTRY["faiss_index"] = faiss.IndexFlatIP(768)
            _REGISTRY["faiss_metadata"] = []
            logger.info("BGE-base-en-v1.5 loaded; empty FAISS index initialised (dim=768)")
    except Exception as exc:
        logger.warning("Could not load BGE + FAISS: %s", exc)


# ---------------------------------------------------------------------------
# Orchestration Layer — load_all / get
# ---------------------------------------------------------------------------

def load_all() -> None:
    """Orchestration Layer: load all models into the registry at startup."""
    if _REGISTRY:
        return  # already loaded
    logger.info("Orchestration Layer: loading all models...")
    _load_deberta_classifier()       # DeBERTa-v3-base  — Ticket Classification
    _load_tfidf_lr_fallback()        # TF-IDF + LR      — Classification Fallback
    _load_priority_lgbm()            # LightGBM Multi   — Priority Prediction
    _load_incident_risk_lgbm()       # LightGBM Binary  — Incident Risk Prediction
    _load_emotion_detector()         # DistilRoBERTa    — Emotion Detection
    _load_minilm_log_ranker()        # MiniLM-L6-v2     — Log Relevance Detection
    _load_bge_faiss()                # BGE + FAISS       — Semantic Retrieval
    logger.info("Model registry ready: %s", list(_REGISTRY.keys()))


def get_model(name: str) -> Any:
    """Retrieve a loaded artefact by name; returns None if unavailable."""
    return _REGISTRY.get(name)


# Load all models
load_all()
print("Model registry:", list(_REGISTRY.keys()) or "(no artifacts found — check ARTIFACTS_DIR)")

## 4. Ticket Classification — DeBERTa-v3-base + TF-IDF/LR Fallback

- Primary: **DeBERTa-v3-base** fine-tuned sequence classifier
- Fallback: **TF-IDF + Logistic Regression** when DeBERTa is unavailable

In [ ]:
"""Ticket classification using DeBERTa-v3-base with TF-IDF + LR fallback."""

import numpy as np


def _build_ticket_text(ticket: dict) -> str:
    """Concatenate ticket fields into a single input string."""
    parts = [
        ticket.get("summary", ""),
        ticket.get("description", ""),
        ticket.get("comments_text", ""),
        ticket.get("top_error_lines", ""),
    ]
    return " [SEP] ".join(p for p in parts if p)


def _classify_with_deberta(text: str) -> str | None:
    """Run DeBERTa-v3-base classifier; return category label or None."""
    import torch

    model = get_model("deberta_model")
    tok = get_model("deberta_tokenizer")
    le = get_model("cat_le")
    if model is None or tok is None:
        return None

    try:
        inputs = tok(
            text,
            return_tensors="pt",
            truncation=True,
            max_length=512,
            padding=True,
        )
        with torch.no_grad():
            logits = model(**inputs).logits
        idx = int(torch.argmax(logits, dim=-1).item())

        # Use label encoder if available (fine-tuned model), else use id2label from config
        if le is not None:
            return str(le.inverse_transform([idx])[0])
        id2label = model.config.id2label
        return str(id2label.get(idx, f"class_{idx}"))
    except Exception as exc:
        logger.warning("DeBERTa classification failed: %s", exc)
        return None


def _classify_with_tfidf_lr(text: str) -> str | None:
    """TF-IDF + Logistic Regression fallback classifier."""
    vectorizer = get_model("tfidf_vectorizer")
    clf = get_model("lr_classifier")
    le = get_model("lr_label_encoder")
    if vectorizer is None or clf is None:
        return None

    try:
        X = vectorizer.transform([text])
        idx = int(clf.predict(X)[0])
        if le is not None:
            return str(le.inverse_transform([idx])[0])
        return str(idx)
    except Exception as exc:
        logger.warning("TF-IDF/LR fallback classification failed: %s", exc)
        return None


def classify_category(ticket: dict) -> str:
    """Classify ticket category.

    Primary: DeBERTa-v3-base
    Fallback: TF-IDF + Logistic Regression
    Default: value from ticket or 'Unknown'

    Returns:
        Category label string.
    """
    text = _build_ticket_text(ticket)

    category = _classify_with_deberta(text)
    if category:
        logger.info("DeBERTa-v3-base classified ticket as: %s", category)
        return category

    category = _classify_with_tfidf_lr(text)
    if category:
        logger.info("TF-IDF/LR fallback classified ticket as: %s", category)
        return category

    default = ticket.get("final_category", "Unknown")
    logger.warning("Both classifiers unavailable — using default: %s", default)
    return default


print("Ticket classification functions defined (DeBERTa-v3-base + TF-IDF/LR fallback).")

## 5. Emotion Detection — DistilRoBERTa-base

Detects the dominant emotion in ticket text to enrich priority and RCA context.

In [ ]:
"""Emotion detection using DistilRoBERTa-base (j-hartmann/emotion-english-distilroberta-base)."""


def detect_emotion(text: str) -> dict:
    """Run DistilRoBERTa-base emotion detection on ticket text.

    Args:
        text: concatenated ticket text.

    Returns:
        dict with ``dominant_emotion`` (str) and ``emotion_scores`` (dict[str, float]).
        Returns empty result if model is unavailable.
    """
    emotion_pipeline = get_model("emotion_pipeline")
    if emotion_pipeline is None:
        logger.warning("DistilRoBERTa emotion detector not available")
        return {"dominant_emotion": "unknown", "emotion_scores": {}}

    try:
        # Truncate to avoid token limit; DistilRoBERTa handles up to 512 tokens
        results = emotion_pipeline(text[:1024])
        # results is a list of lists when top_k=None
        scores_list = results[0] if isinstance(results[0], list) else results
        emotion_scores = {item["label"]: round(item["score"], 4) for item in scores_list}
        dominant_emotion = max(emotion_scores, key=emotion_scores.get)
        logger.info("Emotion detected: %s (scores: %s)", dominant_emotion, emotion_scores)
        return {"dominant_emotion": dominant_emotion, "emotion_scores": emotion_scores}
    except Exception as exc:
        logger.warning("Emotion detection failed: %s", exc)
        return {"dominant_emotion": "unknown", "emotion_scores": {}}


# --- Demo ---
sample_text = "The payment gateway is completely broken and users are furious! This is a critical outage!"
# emotion_result = detect_emotion(sample_text)
# print("Emotion result:", emotion_result)

print("Emotion detection defined (DistilRoBERTa-base).")

## 6. Priority Prediction — LightGBM (Multiclass)

Predicts ticket priority (P1–P5) using LightGBM trained on TF-IDF features, numeric fields, and OHE-encoded categorical fields.

In [ ]:
"""Priority prediction using LightGBM (Multiclass) — P1 to P5."""


def predict_priority(ticket: dict) -> str:
    """Predict ticket priority (P1–P5) using LightGBM Multiclass.

    Feature vector: TF-IDF of text  +  numeric fields  +  OHE categorical fields

    Args:
        ticket: ticket dict.

    Returns:
        Priority label string (e.g. 'P1') or default ('P3') if model unavailable.
    """
    booster = get_model("prio_booster")
    tfidf = get_model("prio_tfidf")
    le = get_model("prio_le")
    if booster is None or tfidf is None or le is None:
        default = ticket.get("final_priority", "P3")
        logger.warning("LightGBM priority model unavailable — using default: %s", default)
        return default

    try:
        text = _build_ticket_text(ticket)
        tfidf_feat = tfidf.transform([text]).toarray()

        numeric_cols = [
            "affected_users", "downtime_minutes", "error_count",
            "fatal_count", "timeout_count", "auth_error_count",
        ]
        num_feat = np.array([[float(ticket.get(c, 0) or 0) for c in numeric_cols]])

        ohe = get_model("prio_ohe")
        cat_cols = ["env", "service"]
        cat_vals = [[str(ticket.get(c, "unknown")) for c in cat_cols]]
        cat_feat = ohe.transform(cat_vals) if ohe is not None else np.zeros((1, 1))

        X = np.hstack([tfidf_feat, num_feat, cat_feat])
        proba = booster.predict(X)          # shape: (1, num_classes)
        idx = int(np.argmax(proba, axis=1)[0])
        priority = str(le.inverse_transform([idx])[0])
        logger.info("LightGBM (Multiclass) predicted priority: %s", priority)
        return priority
    except Exception as exc:
        logger.warning("Priority prediction failed: %s", exc)
        return ticket.get("final_priority", "P3")


print("Priority prediction defined (LightGBM Multiclass).")

## 7. Incident Risk Prediction — LightGBM (Binary)

Predicts the probability [0–1] that a ticket will escalate to a production incident.

In [ ]:
"""Incident risk prediction using LightGBM (Binary classifier)."""


def predict_incident_risk(ticket: dict) -> float:
    """Return probability [0,1] that this ticket precedes a production incident.

    Feature vector: TF-IDF of text  +  numeric fields

    Args:
        ticket: ticket dict.

    Returns:
        Float probability in [0, 1].
    """
    booster = get_model("incident_booster")
    if booster is None:
        logger.warning("LightGBM (Binary) incident risk model unavailable — returning 0.0")
        return 0.0

    try:
        tfidf = get_model("prio_tfidf")  # reuse the same TF-IDF vectorizer
        text = _build_ticket_text(ticket)
        tfidf_feat = tfidf.transform([text]).toarray() if tfidf else np.zeros((1, 100))

        numeric_cols = [
            "affected_users", "downtime_minutes", "error_count",
            "fatal_count", "timeout_count", "auth_error_count",
        ]
        num_feat = np.array([[float(ticket.get(c, 0) or 0) for c in numeric_cols]])

        X = np.hstack([tfidf_feat, num_feat])
        proba = booster.predict(X)   # returns raw score for binary classifier
        risk = float(proba[0]) if proba.ndim == 1 else float(proba[0, 1])
        logger.info("LightGBM (Binary) incident risk score: %.3f", risk)
        return risk
    except Exception as exc:
        logger.warning("Incident risk prediction failed: %s", exc)
        return 0.0


print("Incident risk prediction defined (LightGBM Binary).")

## 8. Log Relevance Detection — MiniLM (all-MiniLM-L6-v2)

Uses **all-MiniLM-L6-v2** embeddings + cosine similarity to rank raw log lines by relevance to the ticket query. Replaces the previous cross-encoder approach.

In [ ]:
"""Log relevance detection using all-MiniLM-L6-v2 (SentenceTransformer)."""

from datetime import datetime, timezone, timedelta


def _rank_logs_with_minilm(query: str, log_lines: list[str], top_k: int) -> list[str]:
    """Embed query and log lines with MiniLM; return top-k by cosine similarity.

    Args:
        query: ticket summary + description.
        log_lines: raw log lines fetched from Azure / Datadog.
        top_k: number of most-relevant lines to return.

    Returns:
        List of the top-k most relevant log lines.
    """
    model = get_model("minilm_model")
    if model is None or not log_lines:
        logger.warning("MiniLM log ranker unavailable — returning first %d lines", top_k)
        return log_lines[:top_k]

    try:
        import numpy as np

        query_emb = model.encode([query], normalize_embeddings=True)          # (1, dim)
        log_embs = model.encode(log_lines, normalize_embeddings=True, batch_size=64)  # (N, dim)

        # Cosine similarity: dot product of normalised embeddings
        scores = (log_embs @ query_emb.T).squeeze()                           # (N,)
        top_indices = np.argsort(scores)[::-1][:top_k]
        ranked = [log_lines[i] for i in top_indices]
        logger.info("MiniLM-L6-v2 ranked %d log lines; returning top %d", len(log_lines), top_k)
        return ranked
    except Exception as exc:
        logger.warning("MiniLM log ranking failed: %s", exc)
        return log_lines[:top_k]


# ---------------------------------------------------------------------------
# Azure File Share — mobile logs
# ---------------------------------------------------------------------------

def _fetch_azure_logs(service: str, env: str, lookback_hours: int = 6) -> list[str]:
    """Fetch log lines from Azure File Share for mobile/app issues."""
    try:
        from azure.storage.fileshare import ShareServiceClient

        if not AZURE_STORAGE_CONNECTION_STRING:
            logger.warning("AZURE_STORAGE_CONNECTION_STRING not set — skipping mobile logs")
            return []

        svc = ShareServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)
        share_client = svc.get_share_client(AZURE_FILE_SHARE_NAME)
        directory = AZURE_LOG_DIRECTORY or service
        dir_client = share_client.get_directory_client(directory)
        cutoff = datetime.now(timezone.utc) - timedelta(hours=lookback_hours)
        lines: list[str] = []

        for item in dir_client.list_directories_and_files():
            if item["is_directory"]:
                continue
            if item.get("last_modified") and item["last_modified"] < cutoff:
                continue
            file_client = dir_client.get_file_client(item["name"])
            content = file_client.download_file().readall().decode("utf-8", errors="replace")
            lines.extend(content.splitlines())

        logger.info("Fetched %d log lines from Azure File Share (service=%s)", len(lines), service)
        return lines
    except Exception as exc:
        logger.error("Azure log fetch failed: %s", exc)
        return []


# ---------------------------------------------------------------------------
# Datadog — server logs
# ---------------------------------------------------------------------------

def _fetch_datadog_logs(service: str, env: str, lookback_hours: int | None = None) -> list[str]:
    """Fetch log lines from Datadog Logs API for server issues."""
    try:
        import requests

        if not DATADOG_API_KEY:
            logger.warning("DATADOG_API_KEY not set — skipping Datadog logs")
            return []

        hours = lookback_hours or DATADOG_LOG_LOOKBACK_HOURS
        now = datetime.now(timezone.utc)
        start = now - timedelta(hours=hours)

        url = f"https://api.{DATADOG_SITE}/api/v2/logs/events/search"
        headers = {
            "DD-API-KEY": DATADOG_API_KEY,
            "DD-APPLICATION-KEY": DATADOG_APP_KEY,
            "Content-Type": "application/json",
        }
        query_filter = f"service:{service}"
        if env:
            query_filter += f" env:{env}"
        query_filter += " status:(error OR warn OR critical)"

        payload = {
            "filter": {
                "query": query_filter,
                "from": start.strftime("%Y-%m-%dT%H:%M:%SZ"),
                "to": now.strftime("%Y-%m-%dT%H:%M:%SZ"),
            },
            "sort": "timestamp",
            "page": {"limit": 1000},
        }

        resp = requests.post(url, json=payload, headers=headers, timeout=20)
        resp.raise_for_status()
        lines = [
            evt.get("attributes", {}).get("message", "")
            for evt in resp.json().get("data", [])
            if evt.get("attributes", {}).get("message", "")
        ]
        logger.info("Fetched %d log lines from Datadog (service=%s)", len(lines), service)
        return lines
    except Exception as exc:
        logger.error("Datadog log fetch failed: %s", exc)
        return []


# ---------------------------------------------------------------------------
# Main log fetch entry point
# ---------------------------------------------------------------------------

def fetch_logs(ticket: dict) -> dict:
    """Fetch logs and detect relevance with all-MiniLM-L6-v2.

    Determines source (Azure = mobile/app, Datadog = server) from ticket category.

    Args:
        ticket: dict with ``key``, ``category``, ``service``, ``env``.

    Returns:
        dict with ``top_log_lines``, ``source``, ``error_count``, ``warn_count``, ``fatal_count``.
    """
    load_all()

    category = (ticket.get("category") or ticket.get("final_category") or "").lower()
    service = ticket.get("service", "")
    env = ticket.get("env", "")
    query = f"{ticket.get('summary', '')} {ticket.get('description', '')}"

    is_mobile = any(kw in category for kw in ("mobile", "android", "ios", "app"))

    if is_mobile:
        raw_lines = _fetch_azure_logs(service, env)
        source = "azure"
    else:
        raw_lines = _fetch_datadog_logs(service, env)
        source = "datadog"

    if not raw_lines:
        return {"top_log_lines": [], "source": "none",
                "error_count": 0, "warn_count": 0, "fatal_count": 0}

    # Use MiniLM-L6-v2 for log relevance detection
    top_lines = _rank_logs_with_minilm(query, raw_lines, TOP_K_LOG_LINES)

    # Threshold monitoring
    error_count = sum(1 for l in raw_lines if "error" in l.lower() or "exception" in l.lower())
    warn_count = sum(1 for l in raw_lines if "warn" in l.lower())
    fatal_count = sum(1 for l in raw_lines if "fatal" in l.lower() or "critical" in l.lower())

    if (
        error_count >= ERROR_COUNT_THRESHOLD
        or warn_count >= WARNING_COUNT_THRESHOLD
        or fatal_count >= FATAL_COUNT_THRESHOLD
    ):
        logger.warning(
            "Log thresholds exceeded for %s — errors=%d, warnings=%d, fatals=%d",
            ticket.get("key"), error_count, warn_count, fatal_count,
        )

    return {
        "top_log_lines": top_lines,
        "source": source,
        "error_count": error_count,
        "warn_count": warn_count,
        "fatal_count": fatal_count,
    }


print("Log relevance detection defined (all-MiniLM-L6-v2).")

## 9. Semantic Retrieval — BGE-base-en-v1.5 + FAISS

Embeds tickets with **BGE-base-en-v1.5** and retrieves similar historical tickets via a **FAISS** inner-product index (RAG-style). Confluence is also searched for KB articles.

In [ ]:
"""Semantic retrieval using BGE-base-en-v1.5 + FAISS (RAG-style)."""

import re
import numpy as np


# ---------------------------------------------------------------------------
# FAISS index helpers
# ---------------------------------------------------------------------------

def index_ticket_in_faiss(ticket: dict) -> None:
    """Embed a resolved ticket with BGE and upsert it into the FAISS index.

    Call this after a ticket is resolved so future queries can retrieve it.

    Args:
        ticket: resolved ticket dict (key, summary, description, category, priority, service).
    """
    embedder = get_model("bge_embedder")
    faiss_index = get_model("faiss_index")
    metadata_store: list = get_model("faiss_metadata") or []

    if embedder is None or faiss_index is None:
        logger.warning("BGE + FAISS not available — skipping ticket indexing")
        return

    try:
        text = (
            f"{ticket.get('summary', '')} "
            f"{ticket.get('description', '')} "
            f"{ticket.get('comments_text', '')}"
        ).strip()
        # BGE recommends prepending "Represent this sentence:" for retrieval tasks
        emb = embedder.encode(
            [f"Represent this sentence: {text}"],
            normalize_embeddings=True,
        ).astype("float32")  # (1, 768)

        faiss_index.add(emb)
        metadata_store.append({
            "key": ticket.get("key", ""),
            "summary": ticket.get("summary", ""),
            "category": ticket.get("category", ""),
            "priority": ticket.get("priority", ""),
            "service": ticket.get("service", ""),
            "text": text[:500],
        })
        logger.info("Indexed ticket %s into FAISS (total: %d)", ticket.get("key"), faiss_index.ntotal)
    except Exception as exc:
        logger.error("Failed to index ticket in FAISS: %s", exc)


def save_faiss_index() -> None:
    """Persist FAISS index and metadata to disk."""
    import faiss

    faiss_index = get_model("faiss_index")
    metadata_store = get_model("faiss_metadata")
    if faiss_index is None:
        return

    import os
    os.makedirs(FAISS_INDEX_PATH, exist_ok=True)
    faiss.write_index(faiss_index, f"{FAISS_INDEX_PATH}/faiss_index.bin")
    joblib.dump(metadata_store, f"{FAISS_INDEX_PATH}/faiss_metadata.joblib")
    logger.info("FAISS index saved to %s", FAISS_INDEX_PATH)


def _search_similar_tickets_faiss(query: str, top_k: int) -> list[dict]:
    """Search FAISS index with BGE-base-en-v1.5 embeddings.

    Args:
        query: ticket summary + description.
        top_k: number of results.

    Returns:
        List of dicts with ``text``, ``metadata``, ``similarity``.
    """
    embedder = get_model("bge_embedder")
    faiss_index = get_model("faiss_index")
    metadata_store: list = get_model("faiss_metadata") or []

    if embedder is None or faiss_index is None or faiss_index.ntotal == 0:
        logger.warning("BGE+FAISS not available or index empty — returning no similar tickets")
        return []

    try:
        # Prepend BGE query prefix for retrieval
        q_emb = embedder.encode(
            [f"Represent this sentence: {query}"],
            normalize_embeddings=True,
        ).astype("float32")  # (1, 768)

        k = min(top_k, faiss_index.ntotal)
        scores, indices = faiss_index.search(q_emb, k)   # inner-product scores

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0 or idx >= len(metadata_store):
                continue
            meta = metadata_store[idx]
            results.append({
                "text": meta.get("text", ""),
                "metadata": meta,
                "similarity": round(float(score), 4),
            })
        logger.info("BGE+FAISS returned %d similar tickets", len(results))
        return results
    except Exception as exc:
        logger.error("FAISS search failed: %s", exc)
        return []


# ---------------------------------------------------------------------------
# Confluence search
# ---------------------------------------------------------------------------

def _search_confluence(query: str, top_k: int) -> list[dict]:
    """Search Confluence using CQL and return top-k matching pages."""
    try:
        import requests
        from requests.auth import HTTPBasicAuth

        if not CONFLUENCE_BASE_URL or not JIRA_API_TOKEN:
            logger.warning("Confluence credentials not configured — skipping")
            return []

        url = f"{CONFLUENCE_BASE_URL}/wiki/rest/api/content/search"
        auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
        cql = (
            f'space = "{CONFLUENCE_SPACE_KEY}" AND text ~ "{query}" '
            f'ORDER BY relevance DESC'
        )
        resp = requests.get(
            url,
            params={"cql": cql, "limit": top_k, "expand": "body.storage,metadata.labels"},
            auth=auth,
            timeout=15,
        )
        resp.raise_for_status()

        pages = []
        for r in resp.json().get("results", []):
            body_val = r.get("body", {}).get("storage", {}).get("value", "")
            plain = re.sub(r"<[^>]+>", " ", body_val)[:1000]
            pages.append({
                "title": r.get("title", ""),
                "url": CONFLUENCE_BASE_URL + r.get("_links", {}).get("webui", ""),
                "excerpt": plain.strip(),
            })
        logger.info("Confluence returned %d pages for query '%s'", len(pages), query[:60])
        return pages
    except Exception as exc:
        logger.error("Confluence search failed: %s", exc)
        return []


# ---------------------------------------------------------------------------
# Combined search context
# ---------------------------------------------------------------------------

def search_context(ticket: dict) -> dict:
    """Retrieve similar tickets (BGE+FAISS) and Confluence articles.

    Args:
        ticket: dict with ``key``, ``summary``, ``description``.

    Returns:
        dict with ``similar_tickets`` (list) and ``confluence_pages`` (list).
    """
    query = f"{ticket.get('summary', '')} {ticket.get('description', '')}".strip()
    if not query:
        return {"similar_tickets": [], "confluence_pages": []}

    similar = _search_similar_tickets_faiss(query, TOP_K_SIMILAR_TICKETS)
    confluence = _search_confluence(query[:200], TOP_K_CONFLUENCE)

    logger.info(
        "Ticket %s: found %d similar tickets, %d Confluence pages",
        ticket.get("key"), len(similar), len(confluence),
    )
    return {"similar_tickets": similar, "confluence_pages": confluence}


print("Semantic retrieval defined (BGE-base-en-v1.5 + FAISS).")

## 10. Rule-Based RCA Generator — Explainable RCA

Generates a structured, explainable Root Cause Analysis using deterministic rule-based logic combined with retrieved evidence (similar tickets, Confluence pages, log lines). This produces transparent, auditable RCA reports without depending solely on LLM black-box output.

In [ ]:
"""Rule-Based RCA Generator — Explainable Root Cause Analysis."""

from typing import Any


# ---------------------------------------------------------------------------
# Rule definitions
# ---------------------------------------------------------------------------

_CATEGORY_ROOT_CAUSE_RULES: dict[str, str] = {
    "network": "Network connectivity or latency degradation between services.",
    "database": "Database query timeout or connection pool exhaustion.",
    "authentication": "Authentication service failure or expired/invalid credentials.",
    "memory": "Memory leak or out-of-memory condition causing process crash.",
    "cpu": "CPU saturation due to inefficient computation or runaway process.",
    "disk": "Disk I/O bottleneck or disk space exhaustion.",
    "deployment": "Regression introduced by recent code deployment or config change.",
    "dependency": "Upstream service or third-party API dependency failure.",
    "mobile": "Client-side crash or API contract mismatch on mobile platform.",
    "timeout": "Request timeout due to slow downstream service or resource contention.",
    "payment": "Payment gateway error or transaction processing failure.",
    "unknown": "Root cause could not be determined from available evidence.",
}

_PRIORITY_IMPACT_MAP: dict[str, str] = {
    "P1": "Critical — full service outage or data loss affecting all users.",
    "P2": "High — major feature degradation affecting a significant user segment.",
    "P3": "Medium — partial feature failure with a workaround available.",
    "P4": "Low — minor degradation with minimal user impact.",
    "P5": "Minimal — cosmetic or non-functional issue.",
}

_EMOTION_RESOLUTION_HINTS: dict[str, str] = {
    "anger": "Prioritise immediate stakeholder communication and provide a status update.",
    "fear": "Escalate to senior engineering and provide a risk mitigation plan.",
    "disgust": "Review quality gates and consider a rollback of recent changes.",
    "sadness": "Conduct a post-mortem and update runbooks to prevent recurrence.",
    "surprise": "Investigate for unexpected configuration drift or undocumented change.",
    "joy": "Incident may be resolved — verify and confirm with stakeholders.",
    "neutral": "Follow standard incident response playbook.",
    "unknown": "Follow standard incident response playbook.",
}


def _match_category_rule(category: str) -> str:
    """Return a root-cause hypothesis based on ticket category."""
    category_lower = category.lower()
    for keyword, hypothesis in _CATEGORY_ROOT_CAUSE_RULES.items():
        if keyword in category_lower:
            return hypothesis
    return _CATEGORY_ROOT_CAUSE_RULES["unknown"]


def _infer_contributing_factors(ticket: dict, log_lines: list[str]) -> list[str]:
    """Derive contributing factors from ticket metrics and log patterns."""
    factors: list[str] = []

    if int(ticket.get("error_count", 0)) >= ERROR_COUNT_THRESHOLD:
        factors.append(f"High error volume ({ticket['error_count']} errors) suggests systemic failure.")
    if int(ticket.get("fatal_count", 0)) >= FATAL_COUNT_THRESHOLD:
        factors.append(f"Fatal events detected ({ticket['fatal_count']}) — possible process crash.")
    if int(ticket.get("timeout_count", 0)) > 0:
        factors.append(f"Timeout events ({ticket['timeout_count']}) indicate downstream latency.")
    if int(ticket.get("affected_users", 0)) > 1000:
        factors.append(f"Wide user impact ({ticket['affected_users']} users) suggests infrastructure-level issue.")
    if int(ticket.get("downtime_minutes", 0)) > 30:
        factors.append(f"Extended downtime ({ticket['downtime_minutes']} min) indicates slow recovery or cascading failure.")

    # Log pattern rules
    log_text = " ".join(log_lines[:50]).lower()
    if "outofmemory" in log_text or "oom" in log_text:
        factors.append("OOM (Out-of-Memory) events detected in logs.")
    if "connection refused" in log_text:
        factors.append("Connection refused errors indicate a service is down or unreachable.")
    if "deadlock" in log_text:
        factors.append("Deadlock detected in logs — database or lock contention issue.")
    if "certificate" in log_text or "ssl" in log_text or "tls" in log_text:
        factors.append("TLS/SSL certificate-related errors detected in logs.")
    if "rate limit" in log_text or "429" in log_text:
        factors.append("Rate limiting detected — external API quota may be exhausted.")

    return factors or ["No significant contributing factors identified from available metrics."]


def _build_resolution_steps(category: str, priority: str, emotion: str) -> list[str]:
    """Generate resolution steps from category + priority + emotion signals."""
    steps: list[str] = []

    if priority in ("P1", "P2"):
        steps.append("Declare incident and assemble on-call response team immediately.")
        steps.append("Enable incident war-room channel and page senior engineers.")

    steps.append(_EMOTION_RESOLUTION_HINTS.get(emotion, _EMOTION_RESOLUTION_HINTS["unknown"]))

    cat_lower = category.lower()
    if "database" in cat_lower:
        steps += [
            "Check database connection pool utilisation and slow-query logs.",
            "Consider scaling read replicas or killing blocking queries.",
        ]
    elif "deployment" in cat_lower:
        steps += [
            "Identify the last deployment or config change.",
            "Roll back to the previous stable release if root cause is confirmed.",
        ]
    elif "network" in cat_lower:
        steps += [
            "Review network topology and check for packet loss / high latency.",
            "Verify DNS resolution and firewall rule changes.",
        ]
    elif "memory" in cat_lower:
        steps += [
            "Capture heap dump and analyse for memory leaks.",
            "Restart affected pods/instances to restore service.",
        ]
    else:
        steps.append("Review service logs and metrics dashboards for anomalies.")
        steps.append("Engage the owning team to investigate the root cause further.")

    steps.append("Update the Jira ticket with findings and set ETA for resolution.")
    return steps


def _format_similar_tickets_section(similar_tickets: list[dict]) -> str:
    """Format similar historical tickets as evidence."""
    if not similar_tickets:
        return "No similar historical tickets found."
    lines = []
    for t in similar_tickets:
        meta = t.get("metadata", {})
        sim = t.get("similarity", 0)
        lines.append(
            f"  - [{meta.get('key', 'N/A')}] (similarity {sim:.0%}) — "
            f"{t.get('text', '')[:200]}"
        )
    return "\n".join(lines)


def generate_rca_report(
    ticket: dict,
    classify_result: dict,
    emotion_result: dict,
    context: dict,
) -> dict:
    """Rule-Based RCA Generator — produce an explainable RCA report.

    Args:
        ticket: original ticket dict.
        classify_result: output of classify_category + predict_priority + predict_incident_risk.
        emotion_result: output of detect_emotion.
        context: output of search_context + fetch_logs.

    Returns:
        dict with ``rca_text`` (str) and ``rca_sections`` (dict of section → content).
    """
    ticket_key = ticket.get("key", "N/A")
    category = classify_result.get("category", "Unknown")
    priority = classify_result.get("priority", "P3")
    incident_risk = classify_result.get("incident_risk", 0.0)
    dominant_emotion = emotion_result.get("dominant_emotion", "unknown")

    similar_tickets = context.get("similar_tickets", [])
    confluence_pages = context.get("confluence_pages", [])
    top_log_lines = context.get("top_log_lines", [])

    # ---- Rule-based analysis ----
    root_cause = _match_category_rule(category)
    contributing_factors = _infer_contributing_factors(ticket, top_log_lines)
    impact = _PRIORITY_IMPACT_MAP.get(priority, _PRIORITY_IMPACT_MAP["P3"])
    resolution_steps = _build_resolution_steps(category, priority, dominant_emotion)

    confluence_refs = (
        "\n".join(
            f"  - [{p.get('title', '')}]({p.get('url', '')})"
            for p in confluence_pages
        )
        or "  No relevant KB articles found."
    )

    top_logs_text = (
        "\n".join(f"  {l}" for l in top_log_lines[:10])
        or "  No log lines retrieved."
    )

    similar_text = _format_similar_tickets_section(similar_tickets)

    prevention = [
        "Add or improve alerting thresholds for this failure mode in Datadog/PagerDuty.",
        "Create or update runbook for this category of incident.",
        "Schedule a blameless post-mortem within 48 hours.",
        "Consider chaos engineering exercise to validate recovery procedures.",
    ]

    sections: dict[str, Any] = {
        "Root Cause": root_cause,
        "Contributing Factors": contributing_factors,
        "Impact": impact,
        "Incident Risk Score": f"{incident_risk:.0%}",
        "Dominant User Emotion": dominant_emotion,
        "Similar Historical Tickets": similar_text,
        "Relevant KB Articles": confluence_refs,
        "Top Log Evidence": top_logs_text,
        "Resolution Steps": resolution_steps,
        "Prevention / Follow-up": prevention,
    }

    # ---- Render text report ----
    def _bullet(items: list[str]) -> str:
        return "\n".join(f"  • {i}" for i in items)

    rca_text = f"""
=============================================================
  EXPLAINABLE ROOT CAUSE ANALYSIS — {ticket_key}
=============================================================
Ticket  : {ticket.get('summary', '')}
Category: {category}   Priority: {priority}   Incident Risk: {incident_risk:.0%}
Service : {ticket.get('service', 'N/A')}   Env: {ticket.get('env', 'N/A')}
Emotion : {dominant_emotion}
-------------------------------------------------------------

1. ROOT CAUSE
{root_cause}

2. CONTRIBUTING FACTORS
{_bullet(contributing_factors)}

3. IMPACT
{impact}

4. SIMILAR HISTORICAL TICKETS (BGE + FAISS)
{similar_text}

5. RELEVANT KNOWLEDGE BASE ARTICLES
{confluence_refs}

6. TOP LOG EVIDENCE (MiniLM-ranked)
{top_logs_text}

7. RESOLUTION STEPS
{_bullet(resolution_steps)}

8. PREVENTION / FOLLOW-UP
{_bullet(prevention)}
=============================================================
Generated by: Rule-Based RCA Generator | AI Ops Automation
Models used : DeBERTa-v3-base, TF-IDF+LR, LightGBM (Multi+Bin),
              DistilRoBERTa-base, MiniLM-L6-v2, BGE-v1.5+FAISS
============================================================="""

    logger.info("Rule-Based RCA generated for ticket %s", ticket_key)
    return {"rca_text": rca_text.strip(), "rca_sections": sections}


print("Rule-Based RCA Generator defined.")

## 11. Jira Helpers

In [ ]:
"""Helpers for posting results back to Jira."""


def _post_jira_comment(ticket_key: str, body: str) -> None:
    """Post a plain-text comment to a Jira issue."""
    import requests
    from requests.auth import HTTPBasicAuth

    if not JIRA_BASE_URL or not JIRA_API_TOKEN:
        logger.warning("Jira credentials not configured — skipping comment post")
        return

    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{ticket_key}/comment"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
    payload = {
        "body": {
            "type": "doc",
            "version": 1,
            "content": [
                {"type": "paragraph", "content": [{"type": "text", "text": body}]}
            ],
        }
    }
    resp = requests.post(url, json=payload, auth=auth, timeout=15)
    if not resp.ok:
        logger.error("Failed to post Jira comment: %s %s", resp.status_code, resp.text)


def _update_jira_priority(ticket_key: str, priority: str) -> None:
    """Update the Jira priority field."""
    import requests
    from requests.auth import HTTPBasicAuth

    if not JIRA_BASE_URL or not JIRA_API_TOKEN:
        return

    priority_map = {"P1": "Highest", "P2": "High", "P3": "Medium", "P4": "Low", "P5": "Lowest"}
    jira_priority = priority_map.get(priority, "Medium")

    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{ticket_key}"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
    resp = requests.put(
        url, json={"fields": {"priority": {"name": jira_priority}}}, auth=auth, timeout=15
    )
    if not resp.ok:
        logger.error("Failed to update Jira priority: %s %s", resp.status_code, resp.text)


def classify_and_post_to_jira(ticket: dict) -> dict:
    """Run full classification pipeline and post summary to Jira.

    Args:
        ticket: ticket dict.

    Returns:
        dict with ``category``, ``priority``, ``incident_risk``, ``dominant_emotion``.
    """
    load_all()
    ticket_key = ticket.get("key", "UNKNOWN")

    _post_jira_comment(
        ticket_key,
        "\U0001f916 AI Ops: Ticket received. Running DeBERTa classification and LightGBM priority prediction…",
    )

    category = classify_category(ticket)
    priority = predict_priority(ticket)
    incident_risk = predict_incident_risk(ticket)
    emotion_result = detect_emotion(_build_ticket_text(ticket))
    dominant_emotion = emotion_result.get("dominant_emotion", "unknown")

    comment = (
        f"\U0001f4ca *Classification Results*\n"
        f"\u2022 Model: DeBERTa-v3-base (fallback: TF-IDF + LR)\n"
        f"\u2022 Category: {category}\n"
        f"\u2022 Priority (LightGBM Multiclass): {priority}\n"
        f"\u2022 Incident Risk (LightGBM Binary): {incident_risk:.0%}\n"
        f"\u2022 Dominant Emotion (DistilRoBERTa): {dominant_emotion}\n"
    )
    _post_jira_comment(ticket_key, comment)
    _update_jira_priority(ticket_key, priority)

    return {
        "category": category,
        "priority": priority,
        "incident_risk": incident_risk,
        "dominant_emotion": dominant_emotion,
    }


print("Jira helpers defined.")

## 12. Notification

In [ ]:
"""Email notifications for threshold alerts, RCA delivery, and Datadog alerts."""

import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText


def _send_email(subject: str, body_html: str, recipients: list[str] | None = None) -> None:
    to_list = recipients or ALERT_EMAIL_RECIPIENTS
    if not to_list:
        logger.warning("No email recipients configured — skipping email")
        return
    if not SMTP_USER or not SMTP_PASSWORD:
        logger.warning("SMTP credentials not configured — skipping email")
        return

    msg = MIMEMultipart("alternative")
    msg["Subject"] = subject
    msg["From"] = SMTP_USER
    msg["To"] = ", ".join(to_list)
    msg.attach(MIMEText(body_html, "html"))

    try:
        with smtplib.SMTP(SMTP_HOST, SMTP_PORT) as server:
            server.ehlo()
            server.starttls()
            server.login(SMTP_USER, SMTP_PASSWORD)
            server.sendmail(SMTP_USER, to_list, msg.as_string())
        logger.info("Email sent: %s -> %s", subject, to_list)
    except Exception as exc:
        logger.error("Failed to send email: %s", exc)
        raise


def send_rca_notification(ticket: dict, rca_text: str) -> None:
    """Email the RCA report to stakeholders."""
    import html as _html

    ticket_key = ticket.get("key", "N/A")
    subject = f"\U0001f4cb AI Ops RCA Ready \u2014 {ticket_key}: {ticket.get('summary', '')[:80]}"
    rca_html = _html.escape(rca_text).replace("\n", "<br>")

    body = f"""
    <html><body>
    <h2>Root Cause Analysis \u2014 {ticket_key}</h2>
    <p><strong>Summary:</strong> {ticket.get('summary', '')}</p>
    <p><strong>Priority:</strong> {ticket.get('priority', '')} |
       <strong>Category:</strong> {ticket.get('category', '')} |
       <strong>Emotion:</strong> {ticket.get('dominant_emotion', 'unknown')}</p>
    <hr/>
    <pre>{rca_html}</pre>
    <hr/>
    <p><em>Generated by AI Ops Automation — Rule-Based RCA Generator</em></p>
    <p><em>Models: DeBERTa-v3-base, TF-IDF+LR, LightGBM (Multi+Binary),
       DistilRoBERTa-base, MiniLM-L6-v2, BGE-v1.5+FAISS</em></p>
    </body></html>
    """
    _send_email(subject, body)


def send_threshold_alert(
    ticket: dict,
    error_count: int,
    warn_count: int,
    fatal_count: int,
    top_lines: list[str],
) -> None:
    """Send email when log thresholds are exceeded."""
    ticket_key = ticket.get("key", "N/A")
    service = ticket.get("service", "unknown")
    env = ticket.get("env", "unknown")
    subject = f"\u26a0\ufe0f AI Ops Alert: Log Threshold Exceeded \u2014 {ticket_key} [{service}/{env}]"
    lines_html = "".join(f"<li><code>{line[:200]}</code></li>" for line in top_lines)

    body = f"""
    <html><body>
    <h2>\U0001f6a8 Log Threshold Alert (MiniLM-ranked evidence)</h2>
    <p><strong>Ticket:</strong> {ticket_key} \u2014 {ticket.get('summary', '')}</p>
    <p><strong>Service:</strong> {service} | <strong>Env:</strong> {env}</p>
    <table border="1" cellpadding="6" cellspacing="0">
      <tr><th>Metric</th><th>Count</th><th>Threshold</th></tr>
      <tr><td>Errors</td><td>{error_count}</td><td>{ERROR_COUNT_THRESHOLD}</td></tr>
      <tr><td>Warnings</td><td>{warn_count}</td><td>{WARNING_COUNT_THRESHOLD}</td></tr>
      <tr><td>Fatals</td><td>{fatal_count}</td><td>{FATAL_COUNT_THRESHOLD}</td></tr>
    </table>
    <h3>Top Log Lines (ranked by all-MiniLM-L6-v2)</h3>
    <ul>{lines_html}</ul>
    <p>Please investigate immediately.</p>
    </body></html>
    """
    _send_email(subject, body)


def send_datadog_alert(alert_payload: dict) -> None:
    """Email stakeholders when Datadog signals a monitor threshold breach."""
    monitor_name = alert_payload.get("monitor_name", "Unknown Monitor")
    metric = alert_payload.get("metric", "")
    value = alert_payload.get("value", "")
    threshold = alert_payload.get("threshold", "")
    screenshot_url = alert_payload.get("snapshot_url", "")
    transition = alert_payload.get("transition", "ALERT")

    subject = f"\U0001f534 Datadog Monitor {transition}: {monitor_name}"
    screenshot_html = f'<p><a href="{screenshot_url}">View Snapshot</a></p>' if screenshot_url else ""

    body = f"""
    <html><body>
    <h2>\U0001f534 Datadog Monitor Alert</h2>
    <p><strong>Monitor:</strong> {monitor_name}</p>
    <p><strong>Metric:</strong> {metric}</p>
    <p><strong>Current Value:</strong> {value} | <strong>Threshold:</strong> {threshold}</p>
    <p><strong>Transition:</strong> {transition}</p>
    {screenshot_html}
    <pre>{str(alert_payload)[:2000]}</pre>
    <p><em>AI Ops has queued an RCA for this event.</em></p>
    </body></html>
    """
    _send_email(subject, body)


print("Notification functions defined.")

## 13. Confluence KB

In [ ]:
"""Create Confluence knowledge base articles from RCA reports."""


def _markdown_to_confluence_storage(md: str) -> str:
    """Convert Markdown to Confluence Storage Format."""
    lines = md.split("\n")
    out: list[str] = []
    for line in lines:
        if line.startswith("### "):
            out.append(f"<h3>{line[4:]}</h3>")
        elif line.startswith("## "):
            out.append(f"<h2>{line[3:]}</h2>")
        elif line.startswith("# "):
            out.append(f"<h1>{line[2:]}</h1>")
        else:
            line = re.sub(r"\*\*(.*?)\*\*", r"<strong>\1</strong>", line)
            if line.startswith("- "):
                out.append(f"<li>{line[2:]}</li>")
            else:
                out.append(f"<p>{line}</p>" if line.strip() else "<p> </p>")
    return "\n".join(out)


def create_kb_article(ticket: dict, rca_text: str) -> dict:
    """Create a Confluence page from the RCA report.

    Args:
        ticket: ticket dict.
        rca_text: RCA text from the Rule-Based RCA Generator.

    Returns:
        dict with ``page_id`` and ``page_url``.
    """
    import requests
    from requests.auth import HTTPBasicAuth

    if not CONFLUENCE_BASE_URL or not JIRA_API_TOKEN:
        logger.warning("Confluence credentials not configured — skipping KB creation")
        return {"page_id": None, "page_url": None}

    ticket_key = ticket.get("key", "N/A")
    summary = ticket.get("summary", "Untitled")
    category = ticket.get("category", "General")
    service = ticket.get("service", "unknown")

    title = f"[RCA] {ticket_key} \u2014 {summary[:80]}"
    storage_body = _markdown_to_confluence_storage(rca_text)

    page_body = f"""
<ac:structured-macro ac:name="info">
  <ac:rich-text-body>
    <p>Auto-generated RCA by AI Ops (Rule-Based RCA Generator) for Jira ticket
       <strong>{ticket_key}</strong> ({category} / {service}).
       Models used: DeBERTa-v3-base, TF-IDF+LR, LightGBM (Multi+Binary),
       DistilRoBERTa-base, MiniLM-L6-v2, BGE-v1.5+FAISS.
    </p>
  </ac:rich-text-body>
</ac:structured-macro>
{storage_body}
"""

    url = f"{CONFLUENCE_BASE_URL}/wiki/rest/api/content"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
    payload: dict = {
        "type": "page",
        "title": title,
        "space": {"key": CONFLUENCE_SPACE_KEY},
        "body": {"storage": {"value": page_body, "representation": "storage"}},
    }
    if CONFLUENCE_PARENT_PAGE_ID:
        payload["ancestors"] = [{"id": CONFLUENCE_PARENT_PAGE_ID}]

    resp = requests.post(url, json=payload, auth=auth, timeout=20)
    resp.raise_for_status()
    data = resp.json()
    page_id = data.get("id")
    page_url = CONFLUENCE_BASE_URL + data.get("_links", {}).get("webui", "")

    logger.info("Created Confluence KB article '%s' (id=%s)", title, page_id)

    # Link Confluence page back to Jira
    _link_confluence_to_jira(ticket_key, page_url, title)

    return {"page_id": page_id, "page_url": page_url}


def _link_confluence_to_jira(ticket_key: str, page_url: str, page_title: str) -> None:
    """Add a remote link on the Jira issue pointing to the Confluence page."""
    import requests
    from requests.auth import HTTPBasicAuth

    if not JIRA_BASE_URL or not JIRA_API_TOKEN:
        return

    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{ticket_key}/remotelink"
    auth = HTTPBasicAuth(JIRA_USER, JIRA_API_TOKEN)
    payload = {
        "globalId": f"confluence-rca-{ticket_key}",
        "object": {
            "url": page_url,
            "title": page_title,
            "icon": {
                "url16x16": "https://confluence.atlassian.com/images/logo/confluence_16.png",
                "title": "Confluence Page",
            },
        },
    }
    resp = requests.post(url, json=payload, auth=auth, timeout=15)
    if not resp.ok:
        logger.warning("Could not link Confluence page to Jira %s: %s", ticket_key, resp.text)


print("Confluence KB functions defined.")

## 14. FastAPI Application

FastAPI endpoints for Jira webhooks, Datadog alerts, and manual pipeline trigger.

In [ ]:
"""FastAPI application: Jira & Datadog webhook endpoints + full pipeline."""

from fastapi import FastAPI, Request, HTTPException
from fastapi.responses import JSONResponse


app = FastAPI(
    title="AI Ops Automation",
    description=(
        "End-to-end AI Ops pipeline using DeBERTa-v3-base, TF-IDF+LR, "
        "LightGBM (Multi+Binary), DistilRoBERTa-base, MiniLM-L6-v2, "
        "BGE-base-en-v1.5+FAISS, and Rule-Based RCA Generator."
    ),
    version="2.0.0",
)


@app.on_event("startup")
async def startup_event() -> None:
    """Pre-load all models via the Orchestration Layer."""
    load_all()
    logger.info("AI Ops application started. Model registry: %s", list(_REGISTRY.keys()))


@app.get("/health", tags=["ops"])
async def health() -> dict:
    return {"status": "ok", "models_loaded": list(_REGISTRY.keys())}


def _extract_description(desc: Any) -> str:
    """Extract plain text from Jira Atlassian Document Format."""
    if desc is None:
        return ""
    if isinstance(desc, str):
        return desc
    parts: list[str] = []

    def _walk(node: Any) -> None:
        if isinstance(node, dict):
            if node.get("type") == "text":
                parts.append(node.get("text", ""))
            for child in node.get("content", []):
                _walk(child)
        elif isinstance(node, list):
            for item in node:
                _walk(item)

    _walk(desc)
    return " ".join(parts).strip()


def _extract_custom_field(fields: dict, field_id: str, default: Any) -> Any:
    val = fields.get(field_id)
    if val is None:
        return default
    if isinstance(val, dict):
        return val.get("value", default)
    return val


def _run_full_pipeline(ticket: dict) -> dict:
    """Execute the complete AI Ops pipeline synchronously.

    Pipeline:
      1. classify_and_post_to_jira (DeBERTa + TF-IDF/LR + LightGBM Multi + Binary + DistilRoBERTa)
      2. fetch_logs (MiniLM-L6-v2 log relevance)
      3. search_context (BGE + FAISS semantic retrieval)
      4. generate_rca_report (Rule-Based RCA Generator)
      5. send_rca_notification + create_kb_article
    """
    # Step 1: Classify
    classify_result = classify_and_post_to_jira(ticket)
    enriched = {**ticket, **classify_result}

    # Step 2: Fetch and rank logs (MiniLM)
    log_result = fetch_logs(enriched)

    # Step 3: Semantic retrieval (BGE + FAISS)
    search_result = search_context(enriched)

    # Step 4: Merge context
    context = {
        **log_result,
        **search_result,
        "incident_risk": classify_result.get("incident_risk", 0.0),
    }

    # Step 5: Rule-Based RCA Generator
    emotion_result = {
        "dominant_emotion": classify_result.get("dominant_emotion", "unknown")
    }
    rca_result = generate_rca_report(enriched, classify_result, emotion_result, context)

    # Post RCA to Jira
    _post_jira_comment(enriched.get("key", ""), rca_result["rca_text"])

    # Step 6: Notify + KB
    send_rca_notification(ticket=enriched, rca_text=rca_result["rca_text"])
    if not search_result.get("confluence_pages"):
        create_kb_article(ticket=enriched, rca_text=rca_result["rca_text"])

    # Index ticket into FAISS for future retrieval
    index_ticket_in_faiss({**enriched, **classify_result})

    return rca_result


@app.post("/webhook/jira", tags=["webhooks"])
async def jira_webhook(request: Request) -> JSONResponse:
    """Receive Jira issue-created / issue-updated webhook events."""
    try:
        payload: dict = await request.json()
    except Exception:
        raise HTTPException(status_code=400, detail="Invalid JSON payload")

    issue = payload.get("issue", {})
    if not issue:
        return JSONResponse({"status": "ignored", "reason": "no issue in payload"})

    fields = issue.get("fields", {})
    ticket: dict = {
        "key": issue.get("key", ""),
        "summary": fields.get("summary", ""),
        "description": _extract_description(fields.get("description")),
        "comments_text": "",
        "top_error_lines": "",
        "env": _extract_custom_field(fields, "customfield_env", "unknown"),
        "service": _extract_custom_field(fields, "customfield_service", "unknown"),
        "affected_users": _extract_custom_field(fields, "customfield_affected_users", 0),
        "downtime_minutes": _extract_custom_field(fields, "customfield_downtime_minutes", 0),
        "error_count": 0, "fatal_count": 0,
        "timeout_count": 0, "auth_error_count": 0,
    }

    logger.info("Received Jira webhook for ticket %s", ticket["key"])
    import asyncio
    asyncio.get_event_loop().run_in_executor(None, _run_full_pipeline, ticket)

    return JSONResponse({"status": "accepted", "ticket_key": ticket["key"]})


@app.post("/webhook/datadog", tags=["webhooks"])
async def datadog_webhook(request: Request) -> JSONResponse:
    """Receive Datadog monitor alert webhook events."""
    try:
        payload: dict = await request.json()
    except Exception:
        raise HTTPException(status_code=400, detail="Invalid JSON payload")

    logger.info("Received Datadog alert: %s", payload.get("monitor_name", "unknown"))
    send_datadog_alert(payload)

    return JSONResponse({"status": "accepted"})


@app.post("/analyze", tags=["api"])
async def analyze_ticket(ticket: dict) -> JSONResponse:
    """Manually trigger the full pipeline for a ticket dict (for testing)."""
    if not ticket.get("key"):
        raise HTTPException(status_code=400, detail="'key' field is required")
    result = _run_full_pipeline(ticket)
    return JSONResponse({"status": "completed", "ticket_key": ticket["key"],
                         "rca_preview": result.get("rca_text", "")[:500]})


print("FastAPI application defined (v2.0.0).")

## 15. Manual Pipeline Trigger — End-to-End Test

Run the complete pipeline with a sample ticket to verify all models work together.

In [ ]:
# Sample ticket for manual testing
sample_ticket = {
    "key": "OPS-123",
    "summary": "Payment service timing out for EU users",
    "description": (
        "Users in the EU region are experiencing 504 gateway timeouts on "
        "/api/payments. Error rate spiked to 15% at 14:30 UTC. Database "
        "connection pool appears saturated. Deployment was pushed 2 hours ago."
    ),
    "comments_text": "On-call: restarted payment pods, issue persists.",
    "top_error_lines": "ERROR: Connection pool exhausted after 30s",
    "env": "production",
    "service": "payment-service",
    "affected_users": 5000,
    "downtime_minutes": 45,
    "error_count": 120,
    "fatal_count": 2,
    "timeout_count": 95,
    "auth_error_count": 0,
}

# ── Step-by-step execution (uncomment each block to run) ──

# STEP 1: Load models (Orchestration Layer)
# load_all()
# print("Models loaded:", list(_REGISTRY.keys()))

# STEP 2: Ticket Classification — DeBERTa-v3-base (+ TF-IDF/LR fallback)
# category = classify_category(sample_ticket)
# print("Category:", category)

# STEP 3: Emotion Detection — DistilRoBERTa-base
# emotion = detect_emotion(_build_ticket_text(sample_ticket))
# print("Emotion:", emotion)

# STEP 4: Priority Prediction — LightGBM (Multiclass)
# priority = predict_priority(sample_ticket)
# print("Priority:", priority)

# STEP 5: Incident Risk — LightGBM (Binary)
# risk = predict_incident_risk(sample_ticket)
# print(f"Incident Risk: {risk:.0%}")

# STEP 6: Log Relevance — all-MiniLM-L6-v2
# log_result = fetch_logs({**sample_ticket, "category": category})
# print("Top log lines:", log_result["top_log_lines"][:3])

# STEP 7: Semantic Retrieval — BGE-base-en-v1.5 + FAISS
# search_result = search_context(sample_ticket)
# print("Similar tickets:", len(search_result["similar_tickets"]))
# print("Confluence pages:", len(search_result["confluence_pages"]))

# STEP 8: Rule-Based RCA Generator
# classify_result = {"category": category, "priority": priority,
#                    "incident_risk": risk, "dominant_emotion": emotion["dominant_emotion"]}
# context = {**log_result, **search_result, "incident_risk": risk}
# rca = generate_rca_report(sample_ticket, classify_result, emotion, context)
# print(rca["rca_text"])

# ── OR run the full pipeline at once ──
# full_result = _run_full_pipeline(sample_ticket)
# print(full_result["rca_text"])

print("Sample ticket ready. Uncomment steps above to run the pipeline.")